# Procter & Gamble Kazakhstan — Import-to-Shelf Logistics Diagnostic

**Purpose:** Test whether warehouse receiving + local product labelling may be a process bottleneck affecting replenishment lead time and simulated shelf stockouts.

> **Critical limitation:** P&G Kazakhstan does not publicly disclose shipment-level warehouse times, stockout rates, or store-level availability data. This project therefore uses a transparent, deterministic *simulated* dataset. It is a demonstration of an analytics method, not a claim about P&G's actual operational performance. The simulation assumptions are not company KPIs.

Public context and URLs are documented in `README.md` and `data_note.pdf`.


## 1. Load raw data and inspect

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

raw = pd.read_csv("data_raw.csv", dtype=str)
print("Raw shape:", raw.shape)
display(raw.head())
print("Exact duplicate rows:", raw.duplicated().sum())


## 2. Clean the data (same steps used to create `data_clean.csv`)

In [ ]:
df = raw.copy()
# Remove exact duplicate rows only; do not deduplicate on person/company names or broad similarity.
df = df.drop_duplicates().copy()

# Normalize categorical text and known synonyms.
for col in ["region", "product_category"]:
    df[col] = df[col].astype("string").str.strip().str.title()
df["transport_mode"] = (df["transport_mode"].astype("string").str.strip().str.lower()
                        .replace({"truck": "road"}).str.title())
df["peak_period"] = df["peak_period"].astype("string").str.strip().str.title()

# Parse mixed date formats (ISO and DD/MM/YYYY).
df["shipment_date"] = pd.to_datetime(df["shipment_date"], format="mixed", dayfirst=True, errors="coerce")

# Convert decimal commas to decimal points, then numeric types.
numeric_cols = ["distance_km", "shipment_units", "customs_clearance_hours",
                "warehouse_wait_hours", "labeling_hours", "processing_hours",
                "total_lead_days", "stockout_flag", "stockout_days",
                "shelf_availability_pct"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype("string").str.replace(",", ".", regex=False), errors="coerce")

# Impossible distances are treated as missing, not as real negative distances.
df.loc[~df["distance_km"].between(100, 3000), "distance_km"] = np.nan
df.loc[~df["shelf_availability_pct"].between(0, 100), "shelf_availability_pct"] = np.nan

# Median imputation for missing continuous descriptive fields only.
for col in ["distance_km", "shelf_availability_pct"]:
    df[col] = df[col].fillna(df[col].median())

# Target and essential fields are never imputed: fail loudly if missing.
assert df["stockout_flag"].notna().all()
assert df["shipment_date"].notna().all()
assert df["record_id"].notna().all()

df["stockout_flag"] = df["stockout_flag"].astype(int)
df["shipment_units"] = df["shipment_units"].astype(int)
df["stockout_days"] = df["stockout_days"].astype(int)
df = df.sort_values("record_id").reset_index(drop=True)

print("Clean shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())
print("Missing values:", int(df.isna().sum().sum()))
display(df.head())
df.to_csv("data_clean.csv", index=False)
print("Saved data_clean.csv")


## 3. Data checks and descriptive statistics

In [ ]:
assert len(df) >= 1000
assert df.shape[1] >= 10
assert df["record_id"].is_unique
assert set(df["transport_mode"].dropna().unique()).issubset({"Rail", "Road"})
print(df[["distance_km","shipment_units","processing_hours","total_lead_days",
          "stockout_days","shelf_availability_pct"]].describe().round(2))
print("Stockout share:", round(df["stockout_flag"].mean()*100, 2), "%")


## 4. Hypothesis test 1 — Does shipment size relate to warehouse + labelling processing time?

- **H0₁:** Mean processing time is the same for shipments above and at/below the median shipment size.
- **H1₁:** Mean processing time differs between the two groups.
- Test: Welch two-sample t-test (two-sided), α = 0.05.


In [ ]:
median_units = df["shipment_units"].median()
high_volume = df.loc[df["shipment_units"] > median_units, "processing_hours"]
lower_volume = df.loc[df["shipment_units"] <= median_units, "processing_hours"]
t_stat, p_value_1 = stats.ttest_ind(high_volume, lower_volume, equal_var=False)
print("Median shipment units:", median_units)
print("High-volume mean processing hours:", high_volume.mean())
print("Lower-volume mean processing hours:", lower_volume.mean())
print("Welch t:", t_stat, "p-value:", p_value_1)
print("Decision:", "Reject H0_1" if p_value_1 < 0.05 else "Do not reject H0_1")


## 5. Hypothesis test 2 — Is simulated stockout incidence associated with processing delays over 24 hours?

- **H0₂:** Stockout share is the same for processing time ≤24 hours and >24 hours.
- **H1₂:** Stockout share differs between the two groups.
- Test: Pearson chi-square test of independence, α = 0.05.


In [ ]:
df["long_processing"] = df["processing_hours"] > 24
table = pd.crosstab(df["long_processing"], df["stockout_flag"])
chi2, p_value_2, dof, expected = stats.chi2_contingency(table)
rates = df.groupby("long_processing")["stockout_flag"].mean()
print(table)
print("Stockout share, <=24h:", rates.loc[False])
print("Stockout share, >24h:", rates.loc[True])
print("Chi-square:", chi2, "df:", dof, "p-value:", p_value_2)
print("Decision:", "Reject H0_2" if p_value_2 < 0.05 else "Do not reject H0_2")


## 6. Interpretation and business recommendation

The two tests evaluate patterns deliberately encoded in a simulated dataset. Small p-values confirm that the analysis code detects the simulated relationships; they do **not** establish that the same relationships exist in P&G's real operations.

Before changing operations, P&G should validate the model using anonymized shipment-level data: arrival, customs release, warehouse receipt, label application, dispatch, store receipt, shelf-audit result, SKU, region, volume, and transport mode. Then run a controlled pilot (e.g., selected warehouse shifts or product groups) and compare lead time and stockout outcomes with a control group.

**Theory of Constraints sequence:** identify receiving/labelling as a candidate constraint → exploit it (pre-stage label files, remove idle time) → subordinate inbound arrivals and dispatch waves to capacity → elevate only if measured utilization/backlog remains high → repeat after the constraint moves.
